<a href="https://colab.research.google.com/github/import-dhruv/AI-Research/blob/main/prefered_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q accelerate bitsandbytes transformers datasets peft trl gradio huggingface_hub

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from datasets import Dataset, DatasetDict
import json
import gradio as gr
from huggingface_hub import login

# Login to Hugging Face to push dataset later
# You need a User Access Token from https://huggingface.co/settings/tokens
# login() # Uncomment and run if you want to push to HF Hub

In [ ]:
model_id = "microsoft/Phi-3-mini-4k-instruct"
# Alternative: "google/gemma-2b-it"

print(f"Loading {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

In [ ]:
prompts = [
    "Explain quantum entanglement in simple terms.",
    "Write a python function to calculate the factorial of a number.",
    "What are the health benefits of apples?",
    "How do I reset my password on Linux?",
    "Write a short poem about AI."
]

SYSTEM_GOOD = "You are a helpful, concise, and accurate assistant."
SYSTEM_BAD = "You are a lazy assistant. Give verbose, uncertain, or slightly incorrect answers."

def generate_response(prompt, system_instruction):
    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": prompt}
    ]
    try:
        output = pipe(messages, max_new_tokens=256, do_sample=True, temperature=0.7)
        return output[0]['generated_text'][-1]['content']
    except Exception as e:
        return f"Error: {str(e)}"


data = []
print("Generating preference pairs...")

for p in prompts:
    print(f"Processing: {p[:30]}...")

    # Generate Chosen (Good)
    # Note: For better quality, you might hardcode 'chosen' or use a stronger API model
    chosen = generate_response(p, SYSTEM_GOOD)

    # Generate Rejected (Bad)
    rejected = generate_response(p, SYSTEM_BAD)

    # Fallback if model fails
    if len(rejected) < 10:
        rejected = chosen + " [This is a duplicated error response]"

    data.append({
        "prompt": p,
        "chosen": chosen,
        "rejected": rejected
    })

print(f"Generated {len(data)} pairs.")

In [ ]:
dataset = Dataset.from_list(data)

# Split into train/test
dataset_dict = DatasetDict({
    "train": dataset.shuffle(seed=42).select(range(int(len(dataset)*0.9))),
    "test": dataset.shuffle(seed=42).select(range(int(len(dataset)*0.9), len(dataset)))
})

print(dataset_dict)
# Example view
print(dataset_dict['train'][0])

In [ ]:
# Save to Google Drive
from google.colab import drive
drive.mount('/content/drive')

dataset_dict.save_to_disk('/content/drive/MyDrive/my_rlhf_dataset')
print("Dataset saved to Google Drive!")

# OR Push to Hugging Face Hub (Requires login in Cell 2)
# dataset_dict.push_to_hub("your-username/your-dataset-name")